In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})

train['Age'] = train['Age'].fillna(train.groupby(['Pclass','Sex'])['Age'].transform('median'))
test['Age'] = test['Age'].fillna(test.groupby(['Pclass','Sex'])['Age'].transform('median'))

test['Fare'] = test['Fare'].fillna(test['Fare'].median())
train['Embarked'] = train['Embarked'].fillna('S')
test['Embarked'] = test['Embarked'].fillna('S')

train = pd.get_dummies(train, columns=['Embarked'])
test = pd.get_dummies(test, columns=['Embarked'])

train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)

train['Title'] = train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test['Title'] = test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

common_titles = ['Mr','Miss','Mrs','Master']
train['Title'] = train['Title'].apply(lambda x: x if x in common_titles else 'Rare')
test['Title'] = test['Title'].apply(lambda x: x if x in common_titles else 'Rare')

train = pd.get_dummies(train, columns=['Title'])
test = pd.get_dummies(test, columns=['Title'])

train_survived = train['Survived']
train, test = train.align(test, join='left', axis=1, fill_value=0)
train['Survived'] = train_survived

features = [col for col in train.columns if col not in ['Survived','Name','Ticket','Cabin']]

X = train[features]
y = train['Survived']
X_test = test[features]

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

submission.to_csv('submission.csv', index=False)

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv
